### Exploring Ollama

In [1]:
from dotenv import load_dotenv

load_dotenv()

import ollama

### Streaming and Thinking

Streaming allows you to render text as it is produced by the model. It is enabled by default through the REST API, but disabled by default in the SDKs.


In [3]:
from ollama import chat

stream = chat(
  model='llama3.2',
  messages=[{'role': 'user', 'content': 'What is 17 × 23?'}],
  stream=True,
)

in_thinking = False
content = ''
thinking = ''
for chunk in stream:
  if chunk.message.thinking:
    if not in_thinking:
      in_thinking = True
      print('Thinking:\n', end='', flush=True)
    print(chunk.message.thinking, end='', flush=True)
    # accumulate the partial thinking 
    thinking += chunk.message.thinking
  elif chunk.message.content:
    if in_thinking:
      in_thinking = False
      print('\n\nAnswer:\n', end='', flush=True)
    print(chunk.message.content, end='', flush=True)
    # accumulate the partial content
    content += chunk.message.content

  # append the accumulated fields to the messages for the next request
  new_messages = [{ "role": 'assistant', "thinking": thinking, "content": content }]

The answer to 17 × 23 is 391.

Thinking-capable models emit a thinking field that separates their reasoning trace from the final answer.
Use this capability to audit model steps, animate the model thinking in a UI, or hide the trace entirely when you only need the final response.

In [4]:
from ollama import chat

stream = chat(
  model='deepseek-r1',
  messages=[{'role': 'user', 'content': 'What is 17 × 23?'}],
  stream=True,
)

in_thinking = False
content = ''
thinking = ''
for chunk in stream:
  if chunk.message.thinking:
    if not in_thinking:
      in_thinking = True
      print('Thinking:\n', end='', flush=True)
    print(chunk.message.thinking, end='', flush=True)
    # accumulate the partial thinking 
    thinking += chunk.message.thinking
  elif chunk.message.content:
    if in_thinking:
      in_thinking = False
      print('\n\nAnswer:\n', end='', flush=True)
    print(chunk.message.content, end='', flush=True)
    # accumulate the partial content
    content += chunk.message.content

  # append the accumulated fields to the messages for the next request
  new_messages = [{ "role": 'assistant', "thinking": thinking, "content": content }]

<think>
First, I need to multiply 17 by 23.

To make the calculation easier, I'll break down 23 into 20 and 3.

Next, I'll multiply 17 by each part separately. 

Multiplying 17 by 20 gives me 340.

Then, multiplying 17 by 3 results in 51.

Finally, I'll add these two products together: 340 plus 51 equals 391.
</think>

**Solution:**  
We need to find the product of \(17\) and \(23\).

Step 1: Break down \(23\) into \(20 + 3\).  
\(17 \times 23 = 17 \times (20 + 3)\)

Step 2: Multiply \(17\) by each part separately.  
- \(17 \times 20 = 340\)  
- \(17 \times 3 = 51\)

Step 3: Add the two results together.  
\(340 + 51 = 391\)

**Final Answer:**  
\(\boxed{391}\)

### Structured Output

It let you enforce a JSON schema on model responses so you can reliably extract structured data, describe images, or keep every reply consistent.

In [5]:
from ollama import chat
from pydantic import BaseModel

class Pet(BaseModel):
  name: str
  animal: str
  age: int
  color: str | None
  favorite_toy: str | None

class PetList(BaseModel):
  pets: list[Pet]

response = chat(
  model='qwen3:4b',
  messages=[{'role': 'user', 'content': 'I have two cats named Luna and Loki of age 2 and 3.'}],
  format=PetList.model_json_schema(),
)

pets = PetList.model_validate_json(response.message.content)
print(pets)

pets=[Pet(name='Luna', animal='cat', age=2, color='silver', favorite_toy='blue feather wand'), Pet(name='Loki', animal='cat', age=3, color='brown with black patches', favorite_toy='stringy rope')]


### Vision

Vision models accept images alongside text so the model can describe, classify, and answer questions about what it sees.

In [7]:
from ollama import chat
# from pathlib import Path

# Pass in the path to the image
path = "image.png"

# You can also pass in base64 encoded image data
# img = base64.b64encode(Path(path).read_bytes()).decode()
# or the raw bytes
# img = Path(path).read_bytes()

response = chat(
  model='gemma3',
  messages=[
    {
      'role': 'user',
      'content': 'What is in this image? Be concise.',
      'images': [path],
    }
  ],
)

print(response.message.content)

A white truck and several cars on a highway.


### Parallel Tool Calling

It supports tool calling (also known as function calling) which allows a model to invoke tools and incorporate their results into its replies.

In [8]:
from ollama import chat

def get_temperature(city: str) -> str:
  """Get the current temperature for a city
  
  Args:
    city: The name of the city

  Returns:
    The current temperature for the city
  """
  temperatures = {
    "New York": "22°C",
    "London": "15°C",
    "Tokyo": "18°C"
  }
  return temperatures.get(city, "Unknown")

def get_conditions(city: str) -> str:
  """Get the current weather conditions for a city
  
  Args:
    city: The name of the city

  Returns:
    The current weather conditions for the city
  """
  conditions = {
    "New York": "Partly cloudy",
    "London": "Rainy",
    "Tokyo": "Sunny"
  }
  return conditions.get(city, "Unknown")


messages = [{'role': 'user', 'content': 'What are the current weather conditions and temperature in New York and London?'}]

# The python client automatically parses functions as a tool schema so we can pass them directly
# Schemas can be passed directly in the tools list as well 
response = chat(model='qwen3.5:2b', messages=messages, tools=[get_temperature, get_conditions], think=True)

# add the assistant message to the messages
messages.append(response.message)
if response.message.tool_calls:
  # process each tool call 
  for call in response.message.tool_calls:
    # execute the appropriate tool
    if call.function.name == 'get_temperature':
      result = get_temperature(**call.function.arguments)
    elif call.function.name == 'get_conditions':
      result = get_conditions(**call.function.arguments)
    else:
      result = 'Unknown tool'
    # add the tool result to the messages
    messages.append({'role': 'tool',  'tool_name': call.function.name, 'content': str(result)})

  # generate the final response
  final_response = chat(model='qwen3.5:2b', messages=messages, tools=[get_temperature, get_conditions], think=True)
  print(final_response.message.content)

Here are the current weather conditions and temperature in New York and London:

**New York:**
- **Temperature:** 22°C
- **Conditions:** Partly cloudy

**London:**
- **Temperature:** 15°C
- **Conditions:** Rainy


### WebSearch

Ollama’s web search API can be used to augment models with the latest information to reduce hallucinations and improve accuracy.

**Web Search API**

Performs a web search for a single query and returns relevant results.

**Web Fetch API**

Fetches a single web page by URL and returns its content.


In [3]:
from ollama import chat, web_fetch, web_search
import os

os.environ["OLLAMA_API_KEY"] = os.getenv("OLLAMA_API_KEY")

available_tools = {'web_search': web_search, 'web_fetch': web_fetch}

messages = [{'role': 'user', 'content': "what is ollama's new engine"}]

while True:
  response = chat(
    model='qwen3:4b',
    messages=messages,
    tools=[web_search, web_fetch],
    think=True
    )
  if response.message.thinking:
    print('Thinking: ', response.message.thinking)
  if response.message.content:
    print('Content: ', response.message.content)
  messages.append(response.message)
  if response.message.tool_calls:
    print('Tool calls: ', response.message.tool_calls)
    for tool_call in response.message.tool_calls:
      function_to_call = available_tools.get(tool_call.function.name)
      if function_to_call:
        args = tool_call.function.arguments
        result = function_to_call(**args)
        print('Result: ', str(result)[:200]+'...')
        # Result is truncated for limited context lengths
        messages.append({'role': 'tool', 'content': str(result)[:2000 * 4], 'tool_name': tool_call.function.name})
      else:
        messages.append({'role': 'tool', 'content': f'Tool {tool_call.function.name} not found', 'tool_name': tool_call.function.name})
  else:
    break

Thinking:  Okay, the user is asking about Ollama's new engine. Let me think. Ollama is a platform for running large language models locally. I remember they've been updating their models recently. The latest version might have a new engine.

First, I should check what Oll. Wait, maybe the user means the latest model release from Ollama. Ollama has models like the "llama3" series. Wait, no, Ollama itself is the platform, and they have different models. Wait, perhaps the user is confused between Ollama (the platform) and the models it supports.

Wait, Ollama's main function is to run models locally, so they don't have their own engine. The engine here might refer to the model they've just released. Let me check recent news.

I need to do a web search to find out what Ollama's latest model or engine is. The user says "new engine", but maybe they mean the latest model. For example, Ollama released a new model called "llama3" or something else.

Let me use the web_search function. The query